# **Initialize Gemini API**

In [1]:
from google import genai

client = genai.Client()
model = "gemini-3.8-flash"

def read_sas():
    try:
        with open("../data/Project_1/SAS Programs/0 - Data and Measure Standardization_2025Jul.sas", "r", encoding='utf8') as file:
            content = file.read()
            return content
    except FileNotFoundError:
        return f"Error: The file path was not found."

output = read_sas()

# **Get Completion Function**
General purpose function to make an LLM call.

Parameters:
- prompt
    - A user's query
- system_message
    - Provide's the LLM with a role
- model
    - Model to use (e.g. gemini 3.8 flash)
- client
    - gemini, openai, etc
- previous_interaction_id
    - Provide the LLM with context from a pervious call

In [2]:
def get_completion(
        prompt: str, 
        system_message: str, 
        model: str, 
        client,
        previous_interaction_id: str | None = None,
    ) -> tuple[str, str]:
    
    response = client.interactions.create(
        model=model,
        system_instruction=system_message,
        input=prompt,
        previous_interaction_id=previous_interaction_id,
    )

    if not response.output_text:
        raise RuntimeError("Model returned with no text.")

    return response.output_text, response.id

In [3]:
prompt = '''
You are tasked with translating this to spanish: Hello
'''

res, id = get_completion(prompt, "You are a helpful assistant.", model, client)

print(res)

**Hola**


# **Reflection Workflow**

## **Initial Translation**

**Initial Translation Function**

Parameters:
- src_code_language
    - The language of the current code
- trg_code_language
    - The language of the code you want to translate too
- src_code
    - The code itself

In [4]:
def initial_translation(
        src_code_language: str,
        trg_code_language: str,
        src_code: str
):
    system_prompt = '''You are an expert in all programming languages. 
        Your task is to translate code from one language to another.
    '''

    translation_prompt = f'''
    Translate the following {src_code_language} code to {trg_code_language}. Preserve all logic, variable names, and 
    functionality. Return only the code block with no extra commentary or details.

    [Source Code]:
    <source_code>
    {src_code}
    <source_code>
    '''

    res, id = get_completion(translation_prompt, system_prompt, model, client)

    return res, id

### **Testing Initial Translation Call**

In [5]:
src_code = '''
def print_hw():
    print("hello, world!")

'''

init_res, init_id = initial_translation("Python", "Java", src_code)

print(init_res)

```java
public class Main {
    public static void print_hw() {
        System.out.println("hello, world!");
    }
}
```


## **Reflection**

In [ ]:
def reflection(
        src_code_language: str,
        trg_code_language: str,
        src_code: str,
        prev_inter_id: str
):
    system_prompt = f'''
        You are an expert in translating {src_code_language} to {trg_code_language}. 
        You will be provided with the source code and its translation. Your goal is to improve the translation.
    '''

    reflection_prompt = f'''
        Your task is to carefully go through each line of code from the source code and the translation,
        and then give constructive criticism and helpful suggestions to improve the translation.
        
        You may assume the source code is correct and no improvements need to be made to the source code.
        
        The overall goal is to ensure that the translated code matches the source code's behavior (1:1).

        NOTE: The translated text resides within the previous interaction.
        The source code, delimited by XML tags <SOURCE_CODE></SOURCE_CODE>, are as follows:

        <SOURCE_CODE>
        {src_code}
        </SOURCE_TEXT>

        When writing suggestions, ensure you account for:
        - Syntax Correctness
            - Attributes and Operations
            - Dependencies
        - Logic / Behavior of the code
        - Output Produced
        - Performance; Time & Space
            - Is the translation slower or faster than the other?
            - Does the translation take more space or less space?
    '''

    res, id = get_completion(reflection_prompt, system_prompt, model, client, prev_inter_id)

    return res, id

### **Testing Reflection Call**

In [7]:
reflection_res, reflection_id = reflection("Python", "Java", src_code, init_id)

print(reflection_res)

Here is a detailed line-by-line review and critique of the translation from Python to Java.

---

### 1. Line-by-Line Comparison & Analysis

| Source Code (Python) | Translation (Java) | Analysis |
| :--- | :--- | :--- |
| *(None - Top-level function)* | `public class Main {` | **Syntax Requirement:** Unlike Python, Java does not support standalone, file-level functions. Wrapping the function inside a class (such as `Main`) is structurally required. |
| `def print_hw():` | `public static void print_hw() {` | **Behavior & Signature:** <br>• In Python, `def print_hw():` with no `return` statement implicitly returns `None`. In Java, `void` is the exact equivalent.<br>• Making the method `static` accurately models a module-level standalone function, allowing it to be invoked without instantiating the class (`Main.print_hw()`).<br>• The name `print_hw` was preserved as requested. |
| `    print("hello, world!")` | `    System.out.println("hello, world!");` | **Output & Logic:** Python's `pr

## **Improved Translation**